# Frontier vs Open Source LLMs: A Practical Comparison

In this notebook, we'll process the same prompt using:
1. **Frontier Model**: OpenAI's GPT-4 via API
2. **Open Source Model**: Meta's Llama via Hugging Face

We'll compare their outputs, costs, and implementation complexity.


| Feature               | Frontier Models (e.g., OpenAI) | Open Source Models (e.g., Llama) |
|-----------------------|--------------------------------|-----------------------------------|
| **Setup Time**        | Minimal (API key)              | Significant (Model download/setup) |
| **Control & Customization** | Limited                      | Full                             |
| **Data Privacy**      | Data sent to external service  | Data stays local                 |
| **Hardware Required** | No (cloud based)               | Yes (GPU recommended)             |
| **Cost**              | Pay per token                  | Pay for compute time             |

## Setup: Install Required Libraries


In [ ]:
%pip install openai transformers accelerate -q


In [ ]:
import os
import time
from openai import OpenAI
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


## Define Our Test Prompt

We'll use the same prompt for both models to compare their responses.


In [ ]:
test_prompt = """Explain what a transformer model is in a long paragraph,
suitable for someone new to machine learning."""

print(f"Our prompt:\n{test_prompt}")


Our prompt:
Explain what a transformer model is in a long paragraph, 
suitable for someone new to machine learning.


## Approach 1: Frontier Model (OpenAI GPT-4)

**Trade-offs:**
- ✅ Zero setup - just API call
- ✅ State-of-the-art performance
- ❌ Costs per token
- ❌ Data sent to external service
- ❌ No customization


In [ ]:
# Set your OpenAI API key (get one at platform.openai.com)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

# Make API call
start_time = time.time()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": test_prompt}
    ],
    max_tokens=150,
    temperature=0.7
)

frontier_time = time.time() - start_time
frontier_response = response.choices[0].message.content

print("=" * 60)
print("FRONTIER MODEL RESPONSE (GPT-4o-mini)")
print("=" * 60)
print(frontier_response)
print(f"\n⏱️  Response time: {frontier_time:.2f} seconds")
input_cost = response.usage.prompt_tokens / 1e6 * 0.15
output_cost = response.usage.completion_tokens / 1e6 * 0.6
print(f"💰 Estimated cost: ~${round(input_cost + output_cost, 4)}")


FRONTIER MODEL RESPONSE (GPT-4o-mini)
A transformer model is a type of neural network architecture that has revolutionized the field of natural language processing (NLP) and many other areas of machine learning. Introduced in a landmark paper in 2017 titled "Attention is All You Need," the transformer model is designed to handle sequential data, such as text, by using a mechanism called "attention." Unlike traditional models that process data in order (like recurrent neural networks), transformers can look at an entire sequence of words or tokens simultaneously, allowing them to capture relationships and dependencies between distant words more effectively. This is achieved through two main components: the encoder and the decoder. The encoder processes the input data and generates a representation that captures its meaning, while the decoder takes that representation and generates the

⏱️  Response time: 1.97 seconds
💰 Estimated cost: ~$0.0001


In [ ]:
response.usage

CompletionUsage(completion_tokens=150, prompt_tokens=28, total_tokens=178, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

## Approach 2: Open Source Model (Llama 3.2)

**Trade-offs:**
- ✅ Full control & customization
- ✅ Data stays private (runs locally)
- ✅ No per-token costs
- ❌ Requires setup & infrastructure
- ❌ Need GPU for reasonable speed
- ❌ You handle scaling & maintenance


In [ ]:
# Load Llama 3.2 1B model (smaller version for demo)
HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

model_name = "meta-llama/Llama-3.2-1B-Instruct"

print("Loading model... (this may take a minute)")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")


Loading model... (this may take a minute)
Model loaded successfully!


In [ ]:
# Format prompt using Llama's chat template
messages = [
    {"role": "user", "content": test_prompt}
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Tokenize and generate
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

start_time = time.time()

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

opensource_time = time.time() - start_time

# Decode response
full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the assistant's response (remove the prompt)
opensource_response = full_output.split("assistant")[-1].strip()

print("=" * 60)
print("OPEN SOURCE MODEL RESPONSE (Llama 3.2)")
print("=" * 60)
print(opensource_response)
print(f"\n⏱️  Response time: {opensource_time:.2f} seconds")
print(f"💰 Cost: $0 (running on your hardware)")


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


OPEN SOURCE MODEL RESPONSE (Llama 3.2)
Imagine you're trying to teach a computer to understand human language, like a teacher teaching a student a new word or concept. A transformer model is a type of artificial intelligence (AI) technique used in natural language processing (NLP) that helps computers learn how to recognize patterns in language. It's like a super-smart teacher that can read and understand the context of a sentence, without needing to understand each word individually. The transformer model works by breaking down the sentence into smaller parts, called "tokens," and then analyzing each token separately. It's like breaking down a sentence into smaller pieces, like individual words, and then studying each piece to understand how they relate to each other. This process is called "self-attention," and it allows

⏱️  Response time: 3.58 seconds
💰 Cost: $0 (running on your hardware)


## Side-by-Side Comparison

Let's compare the key differences we just experienced.


In [ ]:
print("=" * 80)
print(" " * 25 + "COMPARISON SUMMARY")
print("=" * 80)

comparison = f"""
{'Metric':<25} {'Frontier (GPT-4)':<30} {'Open Source (Llama 3.2)'}
{'-'*80}
{'Setup Complexity':<25} {'Simple API call':<30} {'Model download + loading'}
{'Response Time':<25} {f'{frontier_time:.2f}s':<30} {f'{opensource_time:.2f}s'}
{'Cost Model':<25} {'Pay per token':<30} {'Pay for compute time'}
{'Data Privacy':<25} {'Sent to OpenAI':<30} {'Stays local'}
{'Customization':<25} {'Limited':<30} {'Full control'}
{'Model Size':<25} {'Unknown (proprietary)':<30} {'1B parameters'}
"""

print(comparison)

print("\n📊 Key Insights:")
print(f"• Frontier was {'faster' if frontier_time < opensource_time else 'slower'} but costs per request")
print(f"• Open source requires more setup but gives you full control")
print(f"• For high volume (1000s+ requests), open source becomes more cost-effective")


                         COMPARISON SUMMARY

Metric                    Frontier (GPT-4)               Open Source (Llama 3.2)
--------------------------------------------------------------------------------
Setup Complexity          Simple API call                Model download + loading
Response Time             1.97s                          3.58s
Cost Model                Pay per token                  Pay for compute time
Data Privacy              Sent to OpenAI                 Stays local
Customization             Limited                        Full control
Model Size                Unknown (proprietary)          1B parameters


📊 Key Insights:
• Frontier was faster but costs per request
• Open source requires more setup but gives you full control
• For high volume (1000s+ requests), open source becomes more cost-effective


## Key Takeaways

### Frontier Models (API-based)
- **Best for:** Prototyping, low-medium volume, need best performance
- **Costs:** $0.01-0.10 per 1K tokens (varies by model)
- **Pros:** Zero setup, cutting-edge, scalable infrastructure
- **Cons:** Ongoing costs, less control, data privacy concerns

### Open Source Models (Self-hosted)
- **Best for:** High volume, customization, data privacy requirements
- **Costs:** GPU compute time (~$1-5/hour depending on hardware)
- **Pros:** Full control, no per-request cost, private
- **Cons:** Infrastructure management, slower iteration, technical overhead

### The Reality
Most production systems use **both**:
- Frontier models for complex reasoning tasks
- Open source models for high-volume, simpler tasks
- Start with Frontier, migrate to Open Source as you scale

### Cost Break-Even Example
If you process **100,000 requests/month**:
- **Frontier**: ~$300-500/month (depends on tokens)
- **Open Source**: ~$150-300/month (24/7 GPU rental)

The break-even point is typically around **50,000-100,000 requests/month**.
